In [52]:
# Upload CPP_Decoder
# Make sure to have a GPU available
!nvidia-smi
!nvcc --version

Fri May 22 02:07:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [53]:
!pip install numba numpy pyyaml pybind11

In [54]:
!cd /content/tbcc_parallel_decoder/cpp_decoder && mkdir -p build && cd build && cmake .. && make

-- Found pybind11 cmake dir via pip: /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11
-- Found pybind11: /usr/local/lib/python3.12/dist-packages/pybind11/include (found version "3.0.4")
-- Configuring done (0.3s)
-- Generating done (0.0s)
-- Build files have been written to: /content/tbcc_parallel_decoder/cpp_decoder/build
[100%] Built target cpp_tbcc_decoder


In [55]:
%cd /content/tbcc_parallel_decoder
!mkdir -p lib
!nvcc -shared -Xcompiler -fPIC -O3 -gencode arch=compute_75,code=sm_75 combiner.cu -o lib/libtrellis.so

/content/tbcc_parallel_decoder


In [56]:
import sys, os, numpy as np
sys.path.insert(0, os.path.abspath("cpp_decoder/build"))
import cpp_tbcc_decoder
from utils.cuda_driver import launch_combine_cuda

info = cpp_tbcc_decoder.CodeInformation(1, 2, 3, 0, 0, 15, [13, 17])
trellis_obj = cpp_tbcc_decoder.FeedForwardTrellis(info)
nextStates = trellis_obj.getNextStates()  # (8, 2)
outputs_table = trellis_obj.getOutputs()  # (8, 2)


In [57]:
# check all-ones input to decode to all-zeros for cpp decoder
y_ones = [1.0] * 30 

result_ones = decoder.tbcc_decode(y_ones, [])
print("All-ones input -> C++ message:", list(result_ones.message))
print("Expected:                      ", [0] * 15) # should expect 15 bits of 0s decoded 

All-ones input -> C++ message: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Expected:                       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [58]:
y = [1.0] * 30

In [59]:
# testing with a random input y 
# import random

# # Generate a random 30-element sequence of floats between -2.0 and 2.0           
# y = [random.uniform(-2.0, 2.0) for _ in range(30)]

# print("Random Test sequence y:", [round(val, 2) for val in y])

In [60]:
decoder = cpp_tbcc_decoder.LowRateListDecoder(trellis_obj, info, 10000)
result = decoder.tbcc_decode(y, [])
cpp_message = list(result.message)
print("C++ message:", cpp_message)
print("C++ metric:", result.metric)

C++ message: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C++ metric: 0.0


In [61]:
M = 8   # num states
K = 15  # num stages
n = 2   # output bits per stage (rate 1/2)
INF = float('inf')

cost = np.full((K, M, M), INF, dtype=np.float32)

for t in range(K):
    for s in range(M):
        for inp in range(2):
            d = nextStates[s][inp]
            out_dec = outputs_table[s][inp]
            out_bits = [(out_dec >> (n-1-j)) & 1 for j in range(n)]
            bpsk = [1.0 - 2.0*b for b in out_bits]
            metric = sum((y[n*t+j] - bpsk[j])**2 for j in range(n))
            if metric < cost[t][s][d]:
                cost[t][s][d] = metric
                
  
def path_to_message(path, next_states):
    message = []
    for t in range(len(path) - 1):
        s, d = path[t], path[t+1]
        for inp in range(len(next_states[s])):
            if next_states[s][inp] == d:
                message.append(inp)
                break
    return message

In [67]:
current_matrices = cost
tree_argmins = []
tree_shapes = []

while len(current_matrices) > 1:
    n_mats = len(current_matrices)
    num_pairs = n_mats // 2

    left = np.ascontiguousarray(current_matrices[0:2*num_pairs:2])
    right = np.ascontiguousarray(current_matrices[1:2*num_pairs:2])

    combined, argmin = launch_combine_cuda(left, right, M)
    tree_argmins.append(argmin)

    if n_mats % 2 == 1:
        leftover = current_matrices[-1:]
        current_matrices = np.concatenate([combined, leftover], axis=0)
        tree_shapes.append((num_pairs, True))
    else:
        current_matrices = combined
        tree_shapes.append((num_pairs, False))

final_matrix = current_matrices[0]
best_state = int(np.argmin(np.diag(final_matrix))) # take diagonal of argmin matrix 
best_metric = float(final_matrix[best_state, best_state])

print("Parallel best state:", best_state)
print("Parallel metric:", best_metric)


# Reconstruct Tailpath recursively with the saved best state 
def get_path(level, block_idx, start_state, end_state):
    if level < 0:
        return []
    num_pairs, has_leftover = tree_shapes[level]
    if block_idx == num_pairs:
        return get_path(level - 1, block_idx * 2, start_state, end_state)
    mid_state = int(tree_argmins[level][block_idx][start_state][end_state])
    left_path = get_path(level - 1, block_idx * 2, start_state, mid_state)
    right_path = get_path(level - 1, block_idx * 2 + 1, mid_state, end_state)
    return left_path + [mid_state] + right_path

path = [best_state] + get_path(len(tree_shapes) - 1, 0, best_state, best_state) + [best_state]
print("Parallel path:", path)


# verification 
parallel_message = path_to_message(path, nextStates)
print(f"C++ message:      {cpp_message}")
print(f"Parallel message: {parallel_message}")
print(f"Match: {'Matches' if cpp_message == parallel_message else 'Does Not Match'}")

Parallel best state: 0
Parallel metric: 0.0
Parallel path: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C++ message:      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Parallel message: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Match: Matches
